# MCP OAuth 2LO (Client Credentials) Flow

This notebook implements the **two-legged OAuth** (client credentials) flow for MCP servers
that don't require interactive user authorization.

Per the [MCP OAuth Client Credentials extension](https://modelcontextprotocol.io/extensions/auth/oauth-client-credentials),
this is used for server-to-server integrations, background services, CI/CD pipelines, and
daemon processes where no end user is present.

The flow is:
1. Discover the token endpoint (via RFC 9728 PRM → RFC 8414 AS metadata, or manual config)
2. POST `grant_type=client_credentials` with your `client_id` and `client_secret`
3. Use the returned access token to call the MCP endpoint

Some servers (e.g. AWS Bedrock AgentCore) may not implement RFC 9728 discovery at all —
in that case you provide the token endpoint URL directly.

In [ ]:
import requests

In [ ]:
# --- Configuration ---
# MCP endpoint
#AgentCore Gateway — uses 2LO (client_credentials)
mcpURL = "https://bedrock-agent-gateway-dev-gateway-mcp-s3-crud-uyl7xx0mjx.gateway.bedrock-agentcore.us-west-2.amazonaws.com/mcp"

# For servers that don't support RFC 9728 / RFC 8414 discovery,
# set the token endpoint directly. Leave as None to attempt auto-discovery.
manual_token_endpoint = None

# Client credentials (required for 2LO)
client_id = None      # Set your client_id here or enter when prompted
client_secret = None   # Set your client_secret here or enter when prompted

# Scopes to request (space-separated string or list)
scopes = []

### Step 1: Discover the Token Endpoint

We attempt the standard MCP discovery chain:
1. HEAD the MCP endpoint → look for `WWW-Authenticate` with `resource_metadata`
2. Fetch Protected Resource Metadata (RFC 9728) → get `authorization_servers`
3. Fetch AS Metadata (RFC 8414 / OIDC) → get `token_endpoint`

If any step fails (e.g. the server doesn't implement RFC 9728), we fall back to
`manual_token_endpoint`. If that's also `None`, you'll be prompted to enter it.

In [ ]:
import re
from urllib.parse import urlparse

def _is_json_response(resp):
    return resp.ok and 'json' in resp.headers.get('Content-Type', '')

def _build_wellknown(base_url, prefix):
    """Insert /.well-known/{prefix} between host and path (RFC 9728 / RFC 8414)."""
    p = urlparse(base_url)
    path = p.path.rstrip('/')
    return f"{p.scheme}://{p.netloc}/.well-known/{prefix}{path}"

token_endpoint = manual_token_endpoint
resource = None
discovered_scopes = []

if not token_endpoint:
    print("Attempting auto-discovery...\n")

    # --- PRM discovery ---
    response = requests.head(mcpURL)
    www_auth = response.headers.get('WWW-Authenticate', '')
    print(f"HEAD {mcpURL}")
    print(f"  Status: {response.status_code}  WWW-Authenticate: {www_auth or '(none)'}")

    match = re.search(r'resource_metadata="([^"]+)"', www_auth)
    prm_urls = []
    if match:
        prm_urls.append(match.group(1))
    fallback = _build_wellknown(mcpURL, 'oauth-protected-resource')
    if fallback not in prm_urls:
        prm_urls.append(fallback)

    prm = None
    for url in prm_urls:
        print(f"\nGET {url}")
        r = requests.get(url)
        print(f"  Status: {r.status_code}  Content-Type: {r.headers.get('Content-Type', '(none)')}")
        if _is_json_response(r):
            prm = r.json()
            print(f"  ✓ PRM found")
            break
        print(f"  ✗ Not valid JSON PRM")

    if prm:
        resource = prm.get('resource')
        discovered_scopes = prm.get('scopes_supported', [])
        auth_servers = prm.get('authorization_servers', [])
        print(f"\nResource: {resource}")
        print(f"Authorization Servers: {auth_servers}")

        # --- AS metadata discovery ---
        if auth_servers:
            issuer = auth_servers[0]
            for suffix in ['oauth-authorization-server', 'openid-configuration']:
                as_url = _build_wellknown(issuer, suffix)
                print(f"\nGET {as_url}")
                r = requests.get(as_url)
                ct = r.headers.get('Content-Type', '(none)')
                print(f"  Status: {r.status_code}  Content-Type: {ct}")
                if _is_json_response(r):
                    as_meta = r.json()
                    token_endpoint = as_meta.get('token_endpoint')
                    print(f"  ✓ token_endpoint: {token_endpoint}")
                    break
                print(f"  ✗ Skipping")
    else:
        print("\n⚠ PRM discovery failed — server may not implement RFC 9728.")

if not token_endpoint:
    print("\nAuto-discovery did not find a token endpoint.")
    token_endpoint = input("Enter token_endpoint URL manually: ").strip()

# Use discovered scopes if none were configured
if not scopes and discovered_scopes:
    scopes = discovered_scopes

print(f"\n✓ Token Endpoint: {token_endpoint}")
if resource:
    print(f"✓ Resource: {resource}")
if scopes:
    print(f"✓ Scopes: {scopes}")

### Step 2: Client Credentials

Provide your `client_id` and `client_secret`. These are typically obtained by registering
an application with the authorization server (e.g. via a developer console or admin API).

In [ ]:
if not client_id:
    client_id = input("client_id: ").strip()
if not client_secret:
    client_secret = input("client_secret: ").strip()

print(f"client_id: {client_id}")
print(f"client_secret: {'*' * len(client_secret)}")

### Step 3: Token Request (Client Credentials Grant)

POST to the token endpoint with `grant_type=client_credentials`.
The `resource` parameter (RFC 8707) binds the token to the MCP server if available.

In [ ]:
token_payload = {
    "grant_type": "client_credentials",
    "client_id": client_id,
    "client_secret": client_secret,
}

if scopes:
    scope_str = " ".join(scopes) if isinstance(scopes, list) else scopes
    token_payload["scope"] = scope_str

if resource:
    token_payload["resource"] = resource

print(f"POST {token_endpoint}")
print(f"  grant_type: client_credentials")
if scopes:
    print(f"  scope: {token_payload.get('scope')}")
if resource:
    print(f"  resource: {resource}")

token_response = requests.post(
    token_endpoint,
    data=token_payload,
    headers={"Content-Type": "application/x-www-form-urlencoded"}
)

print(f"\n  Status: {token_response.status_code}")
if not token_response.ok:
    print(f"  Error: {token_response.text}")
    token_response.raise_for_status()

token_data = token_response.json()
access_token = token_data['access_token']
expires_in = token_data.get('expires_in')
token_type = token_data.get('token_type', 'Bearer')

print(f"\n✓ Token type: {token_type}")
print(f"✓ Access token: {access_token[:20]}...")
print(f"✓ Expires in: {expires_in}s")

### Step 4: Test the MCP Endpoint

Make an authenticated request to the MCP server using the access token.

In [ ]:
headers = {
    "Authorization": f"{token_type} {access_token}",
    "Content-Type": "application/json"
}

# MCP initialize request (JSON-RPC)
init_payload = {
    "jsonrpc": "2.0",
    "id": 1,
    "method": "initialize",
    "params": {
        "protocolVersion": "2025-03-26",
        "capabilities": {},
        "clientInfo": {
            "name": "mcp-notebook-2lo",
            "version": "0.1.0"
        }
    }
}

print(f"POST {mcpURL}")
mcp_response = requests.post(mcpURL, json=init_payload, headers=headers)
print(f"  Status: {mcp_response.status_code}")
print(f"  Content-Type: {mcp_response.headers.get('Content-Type', '(none)')}")

if mcp_response.ok:
    import json
    print(f"\nResponse:\n{json.dumps(mcp_response.json(), indent=2)}")
else:
    print(f"\nError: {mcp_response.text}")

### List Available Tools

Send `tools/list` (JSON-RPC) to retrieve all tools the MCP server exposes,
along with their descriptions and input schemas.

In [ ]:
import json

tools_payload = {
    "jsonrpc": "2.0",
    "id": 2,
    "method": "tools/list",
    "params": {}
}

print(f"POST {mcpURL}")
tools_response = requests.post(mcpURL, json=tools_payload, headers=headers)
print(f"  Status: {tools_response.status_code}\n")

tools_data = tools_response.json()

if 'error' in tools_data:
    print(f"Error: {tools_data['error']}")
else:
    tools = tools_data.get('result', {}).get('tools', [])
    print(f"Found {len(tools)} tool(s):\n")
    for tool in tools:
        name = tool.get('name', '(unnamed)')
        desc = tool.get('description', '(no description)')
        print(f"  ▸ {name}")
        print(f"    {desc}")
        schema = tool.get('inputSchema')
        if schema:
            props = schema.get('properties', {})
            required = schema.get('required', [])
            if props:
                print(f"    Parameters:")
                for pname, pdef in props.items():
                    req = ' (required)' if pname in required else ''
                    ptype = pdef.get('type', '?')
                    pdesc = pdef.get('description', '')
                    print(f"      - {pname}: {ptype}{req} — {pdesc}")
        print()